In [4]:
# ===================================================================
#      JAX (Brax PPO) to ONNX Conversion Script (Corrected & Final)
# ===================================================================

# --- 導入所有必要的函式庫 ---
import pickle
import numpy as np
import jax
import tensorflow as tf
from tensorflow.keras import layers
import tf2onnx
import onnx
from etils import epath
import os

# 為了載入 Orbax checkpoint
from orbax import checkpoint as ocp

# --- 步驟 0: 配置 (請根據您的訓練進行修改) ---

# 【請修改】指向您 PPO 訓練的 Checkpoint 主目錄
# 這應該是包含了 '100000', '2000000' 等子目錄的那個'checkpoints'目錄
CHECKPOINT_DIR = epath.Path("checkpoints/PupperJoystickFlatTerrain").resolve() 

# 【請修改】您 Pupper 環境的觀測和動作維度
# 假設 PPO 使用的是字典觀測，且您只關心 'state' 部分
POLICY_OBS_SIZE = 48 
ACTION_SIZE = 12

# 【請修改】您 PPO 訓練時使用的網路結構
POLICY_HIDDEN_LAYER_SIZES = (512, 256, 128)


# --- 步驟 1: 使用 Orbax 正確地載入最新的 JAX 參數 ---
print(f"--- Step 1: Loading JAX parameters from Orbax checkpoint ---")
print(f"Searching for checkpoints in: {CHECKPOINT_DIR}")

# 聲明變數，以防載入失敗
normalizer_params = None
policy_params = None
value_params = None
latest_step = None

try:
    if not CHECKPOINT_DIR.exists() or not any(CHECKPOINT_DIR.iterdir()):
        raise FileNotFoundError(f"Checkpoint directory is empty or does not exist: {CHECKPOINT_DIR}")

    # 1. 查找最新的步數
    steps = [int(p.name) for p in CHECKPOINT_DIR.iterdir() if p.is_dir() and p.name.isdigit()]
    if not steps:
        raise FileNotFoundError(f"No valid checkpoint steps found in: {CHECKPOINT_DIR}")
    latest_step = max(steps)
    print(f"  - Found latest checkpoint at step: {latest_step}")

    # 2. 構建最新的 checkpoint 目錄路徑
    latest_checkpoint_path = CHECKPOINT_DIR / str(latest_step)
    print(f"  - Restoring from: {latest_checkpoint_path}")

    # 3. 使用正確的 Orbax API 來恢復
    checkpointer = ocp.PyTreeCheckpointer()
    params_tuple = checkpointer.restore(str(latest_checkpoint_path))
    
    # 4. 分離參數 (Brax PPO 保存的元組結構是 3 個元素)
    normalizer_params, policy_params, value_params = params_tuple
    
    print("  - JAX parameters loaded and separated successfully.")

except FileNotFoundError as e:
    print(f"  [FATAL ERROR] Could not load checkpoint. Please check your path and ensure the checkpoint is valid. Error: {e}")
    # 在錯誤後停止執行，因為後續步驟無法進行
    raise
except Exception as e:
    print(f"  [FATAL ERROR] An unexpected error occurred while loading Orbax checkpoint: {e}")
    raise


# --- 步驟 2: 創建一個等效的、端到端的 TensorFlow Keras 模型 ---
print("\n--- Step 2: Building equivalent TensorFlow Keras model ---")

# 將觀測標準化作為一個自定義的 Keras 層
class NormalizationLayer(layers.Layer):
    def __init__(self, mean, std, name='normalization_layer', **kwargs):
        super().__init__(name=name, **kwargs)
        # 將 JAX 陣列轉換為 TensorFlow 常量
        self.mean = tf.constant(mean, dtype=tf.float32)
        self.std = tf.constant(std, dtype=tf.float32)

    def call(self, inputs):
        # 模型的輸入是一個字典，我們只關心 'state'
        state_input = inputs['state']
        # 執行標準化
        return (state_input - self.mean) / (self.std + 1e-8)

# 定義與 Brax MLP 結構對應的 Keras MLP 模型
class KerasMLP(tf.keras.Model):
    def __init__(self, layer_sizes, activation=tf.nn.relu, name='mlp_block'):
        super().__init__(name=name)
        self.mlp_layers = []
        for i, size in enumerate(layer_sizes):
            # 只有最後一層的輸出沒有激活函數
            act = activation if i < len(layer_sizes) - 1 else None
            self.mlp_layers.append(layers.Dense(
                size, activation=act, kernel_initializer='lecun_uniform', name=f"hidden_{i}"
            ))
    def call(self, inputs):
        x = inputs
        for layer in self.mlp_layers:
            x = layer(x)
        return x

# 定義工廠函式來創建完整的策略網路
def make_tf_policy_network(policy_obs_size, act_size, hidden_sizes, normalizer_mean, normalizer_std):
    """
    Creates the complete, end-to-end TensorFlow policy network.
    This version uses a Keras Lambda layer for the split operation.
    """
    inputs = {'state': tf.keras.Input(shape=(policy_obs_size,), name='state')}
    
    # 步驟 2a: 標準化
    normalized_obs = NormalizationLayer(normalizer_mean, normalizer_std)(inputs)
    
    # 步驟 2b: MLP
    mlp = KerasMLP(layer_sizes=list(hidden_sizes) + [act_size * 2])
    logits = mlp(normalized_obs)
    
    # 步驟 2c: 【關鍵修正】使用 Keras Lambda 層來包裝 tf.split
    # 我們需要定義一個 lambda 函數，它接收 logits 作為輸入，並返回拆分後的第一部分
    loc_lambda = layers.Lambda(lambda x: tf.split(x, num_or_size_splits=2, axis=-1)[0])
    
    # 將 logits 傳遞給這個 Lambda 層
    loc = loc_lambda(logits)
    
    # 步驟 2d: 最終的激活函數
    outputs = tf.keras.layers.Activation('tanh', name='action')(loc)
    
    return tf.keras.Model(inputs=inputs, outputs=outputs, name="PupperPPOPolicy")

# 【關鍵修正】: 使用方括號 [] 而不是點 . 來訪問從 Orbax 恢復的字典
print("Extracting mean and std from loaded normalizer parameters...")
try:
    mean = np.array(normalizer_params['mean']['state'])
    std = np.array(normalizer_params['std']['state'])
    print("  - Mean and Std extracted successfully.")
except KeyError as e:
    print(f"  [ERROR] Key not found in normalizer_params: {e}")
    raise

# 創建 TF 模型
tf_policy_network = make_tf_policy_network(
    policy_obs_size=POLICY_OBS_SIZE,
    act_size=ACTION_SIZE,
    hidden_sizes=POLICY_HIDDEN_LAYER_SIZES,
    normalizer_mean=mean,
    normalizer_std=std
)
print("  - TensorFlow Keras model with normalization layer built successfully.")
tf_policy_network.summary()

# --- 步驟 3: 將 JAX 權重手動轉移到 TensorFlow 模型 (最終修正版) ---
print("\n--- Step 3: Transferring JAX weights to TensorFlow model ---")

def transfer_weights(jax_policy_params, tf_model):
    """
    Transfers weights from a JAX parameter pytree to a TensorFlow model,
    based on the observed structure {'params': ...}.
    """
    # 【關鍵修正】: 根據診斷結果，直接從正確的路徑提取權重字典
    # jax_policy_params 的結構是 {'params': {'hidden_0': ...}}
    print("  - Accessing weights from jax_policy_params['params']...")
    jax_weights_dict = jax_policy_params['params']
    
    tf_mlp_layers = tf_model.get_layer('mlp_block').layers
    
    i = 0
    # 遍歷 TensorFlow 模型的 Dense 層
    for layer in tf_mlp_layers:
        if isinstance(layer, layers.Dense):
            # 構建對應的 JAX 層的名稱
            layer_name_jax = f"hidden_{i}"
            
            if layer_name_jax not in jax_weights_dict:
                print(f"  [WARNING] Weight key '{layer_name_jax}' not found in JAX params. Skipping layer '{layer.name}'.")
                continue

            # 從 JAX 參數中提取 kernel 和 bias
            jax_layer_params = jax_weights_dict[layer_name_jax]
            kernel = np.array(jax_layer_params['kernel'])
            bias = np.array(jax_layer_params['bias'])
            
            print(f"  - Transferring to TF layer '{layer.name}': kernel{kernel.shape}, bias{bias.shape}")
            
            # 將權重設置到 TensorFlow 層中
            layer.set_weights([kernel, bias])
            i += 1

# 執行權重轉移
# 傳遞我們從元組中分離出來的 `policy_params`
transfer_weights(policy_params, tf_policy_network)
print("  - Weight transfer complete.")


# --- 步驟 4: 將 TensorFlow 模型轉換為 ONNX ---
print("\n--- Step 4: Converting TensorFlow model to ONNX ---")
output_path = f"pupper_ppo_policy_{latest_step}_normalized.onnx"

# 為 Keras 模型定義輸入簽名
# 注意：輸入是一個字典，只包含 'state'
spec = ({'state': tf.TensorSpec((None, POLICY_OBS_SIZE), tf.float32, name="state")},)

try:
    # 進行轉換
    model_proto, _ = tf2onnx.convert.from_keras(
        tf_policy_network, 
        input_signature=spec, 
        opset=13, # 一個常用的穩定版本
        output_path=output_path
    )

    print(f"\nConversion successful! ONNX model saved to: {output_path}")

    # 檢查 ONNX 模型
    print("Checking the ONNX model...")
    onnx.checker.check_model(output_path)
    print("ONNX model checked successfully.")

except Exception as e:
    print(f"  [ERROR] An error occurred during ONNX conversion: {e}")

--- Step 1: Loading JAX parameters from Orbax checkpoint ---
Searching for checkpoints in: /mnt/d/project_pupper/mujoco/mujoco_playground_recoil/mujoco_playground/_src/locomotion/pupper/checkpoints/PupperJoystickFlatTerrain
  - Found latest checkpoint at step: 200540160
  - Restoring from: /mnt/d/project_pupper/mujoco/mujoco_playground_recoil/mujoco_playground/_src/locomotion/pupper/checkpoints/PupperJoystickFlatTerrain/200540160
  - JAX parameters loaded and separated successfully.

--- Step 2: Building equivalent TensorFlow Keras model ---
Extracting mean and std from loaded normalizer parameters...
  - Mean and Std extracted successfully.
  - TensorFlow Keras model with normalization layer built successfully.


Model: "PupperPPOPolicy"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ state (InputLayer)              │ (None, 48)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ normalization_layer             │ (None, 48)             │             0 │
│ (NormalizationLayer)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mlp_block (KerasMLP)            │ (None, 24)             │       192,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_2 (Lambda)               │ (None, 12)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ action (Activation)             │ (None, 12)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 192,408 (751.59 KB)

 Trainable params: 192,408 (751.59 KB)

 Non-trainable params: 0 (0.00 B)


--- Step 3: Transferring JAX weights to TensorFlow model ---
  - Accessing weights from jax_policy_params['params']...
  - Transferring to TF layer 'hidden_0': kernel(48, 512), bias(512,)
  - Transferring to TF layer 'hidden_1': kernel(512, 256), bias(256,)
  - Transferring to TF layer 'hidden_2': kernel(256, 128), bias(128,)
  - Transferring to TF layer 'hidden_3': kernel(128, 24), bias(24,)
  - Weight transfer complete.

--- Step 4: Converting TensorFlow model to ONNX ---


I0000 00:00:1752137318.690229   79382 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1752137318.690467   79382 single_machine.cc:374] Starting new session
I0000 00:00:1752137318.691222   79382 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2173 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9
I0000 00:00:1752137318.775258   79382 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2173 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9
I0000 00:00:1752137318.790489   79382 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1752137318.790753   79382 single_machine.cc:374] Starting new session
I0000 00:00:1752137318.791411   79382 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:


Conversion successful! ONNX model saved to: pupper_ppo_policy_200540160_normalized.onnx
Checking the ONNX model...
ONNX model checked successfully.
